# Ablation 3 — ChatTS-8B vs ChatTS-14B Model Size Comparison

**Thesis Table**: Section 5.x — Model capacity and anomaly classification

Runs Approach 3 (Hybrid: pre-screen + stats + MCQ) with both ChatTS-8B and ChatTS-14B
on Reactor 1 signals and compares accuracy and per-class F1.

Key questions:
- Does the larger 14B model improve MCQ answer quality?
- Which anomaly types benefit most from extra capacity?
- Is the 8B model a viable lower-cost alternative?

**VRAM note**: 14B requires ~75 GiB (A100 80GB), 8B requires ~40 GiB (fits 2× on one A100).

In [ ]:
import sys
sys.path.insert(0, '../..')
from dotenv import load_dotenv
load_dotenv('../../.env')

import torch
from src.data.timeseer_client import fetch_series_api, list_series_api
from src.data.ground_truth import GROUND_TRUTH
from src.models.chatts_loader import load_model
from src.prescreener.analyze import hybrid_analyze
from src.inference.chatts_infer import ask_chatts_chunk
from src.parsing.extract import extract_category
from src.evaluation.metrics import compute_metrics
from src.evaluation.report import print_summary_table

AREA = 'Reactor 1'
MODELS_TO_TEST = ['ChatTS-8B', 'ChatTS-14B']

print('Imports OK')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
tags = list_series_api(AREA)
print(f'{len(tags)} signals in {AREA}')

# Pre-fetch all signals once
signal_cache = {}
for tag in tags:
    vals, idx = fetch_series_api(tag, AREA)
    if vals is not None:
        signal_cache[tag] = (vals, idx)
print(f'Cached {len(signal_cache)} signals')

In [ ]:
all_results = {}

for model_name in MODELS_TO_TEST:
    print(f'\n{"="*55}')
    print(f'Loading {model_name}...')
    model, tokenizer, processor = load_model(model_name)
    torch.cuda.empty_cache()

    results = []
    for tag in tags:
        if tag not in signal_cache:
            continue
        vals, idx = signal_cache[tag]

        chunk, question, tname, detected, chunk_desc = hybrid_analyze(vals, idx, tag)
        torch.cuda.empty_cache()
        answer = ask_chatts_chunk(
            chunk, question,
            model=model, tokenizer=tokenizer, processor=processor
        )
        cat, lbl = extract_category(answer, detected)
        gt = GROUND_TRUTH.get(tag, '?')
        results.append({'Tag': tag, 'gt': gt, 'Category': cat, 'Label': lbl})
        print(f'  {tag:<30} → {cat}) {lbl}  (GT: {gt})')

    all_results[model_name] = results

    # Free model before loading next
    del model, tokenizer, processor
    torch.cuda.empty_cache()

In [ ]:
import numpy as np

print('\n' + '='*65)
print(f'MODEL COMPARISON — {AREA} — Approach 3 (Hybrid)')
print('='*65)

for model_name, results in all_results.items():
    evaluated = [r for r in results if r['gt'] != '?']
    preds = [r['Category'] for r in evaluated]
    gts   = [r['gt']       for r in evaluated]
    m = compute_metrics(preds, gts)
    print(f'\n{model_name}')
    print(f'  Accuracy : {m["accuracy"]:.3f}  ({sum(p==g for p,g in zip(preds,gts))}/{len(gts)})')
    print(f'  Macro F1 : {m["f1"]:.3f}')

print('\n' + '-'*65)
print(f'{"Tag":<30} {"GT":<5}', end='')
for m in MODELS_TO_TEST:
    print(f'  {m:<14}', end='')
print()
print('-'*65)

for tag in tags:
    gt = GROUND_TRUTH.get(tag, '?')
    if gt == '?':
        continue
    print(f'{tag:<30} {gt:<5}', end='')
    for model_name in MODELS_TO_TEST:
        res = next((r for r in all_results[model_name] if r['Tag'] == tag), None)
        pred = res['Category'] if res else '?'
        match = '✓' if pred == gt else '✗'
        print(f'  {pred} {match:<12}', end='')
    print()

In [ ]:
# Per-class breakdown: where does 14B outperform 8B?
from collections import defaultdict

label_names = {'A':'Drift','B':'Spikes','C':'Frozen','D':'Phase','E':'Clean',
               'G':'VarCollapse','L':'Intermittent'}

print('\nPer-class accuracy (fraction correct):')
print(f'{"Class":<15}', end='')
for m in MODELS_TO_TEST:
    print(f'  {m:<14}', end='')
print()
print('-'*50)

all_gts = [r['gt'] for r in all_results[MODELS_TO_TEST[0]] if r['gt'] != '?']
unique_classes = sorted(set(all_gts))

for cls in unique_classes:
    name = label_names.get(cls, cls)
    print(f'{cls}) {name:<12}', end='')
    for model_name in MODELS_TO_TEST:
        cls_results = [r for r in all_results[model_name]
                       if r['gt'] == cls]
        if not cls_results:
            print(f'  {"N/A":<14}', end='')
            continue
        correct = sum(r['Category'] == cls for r in cls_results)
        print(f'  {correct}/{len(cls_results)} ({100*correct/len(cls_results):.0f}%)  ', end='')
    print()